# 04 高斯型求积

高斯型求积使用非等距节点，以尽可能少的函数值获得高代数精度。它的理论基础是正交多项式：对给定权函数，取相应正交多项式的零点作为节点，可以使 $n$ 点公式达到 $2n-1$ 次代数精度。

本节重点完整实现 Gauss-Legendre 求积，并给出 Gauss-Chebyshev、Gauss-Laguerre 和 Gauss-Hermite 的带权积分对照。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    composite_simpson,
    gauss_chebyshev_integrate,
    gauss_hermite_integrate,
    gauss_laguerre_integrate,
    gauss_legendre_integrate,
    gauss_legendre_nodes_weights,
)


## 带权积分和正交多项式

高斯型求积处理的基本对象是

$$
I(f)=\int_a^b f(x)w(x)\,dx.
$$

若多项式序列 $p_0,p_1,\ldots$ 关于权函数 $w$ 正交，即

$$
\int_a^b p_i(x)p_j(x)w(x)\,dx=0,\qquad i\ne j,
$$

则 $n$ 点 Gauss 公式选取 $p_n$ 的 $n$ 个零点为节点。它可以精确积分所有次数不超过 $2n-1$ 的多项式。

常见对应关系：

| 公式 | 区间 | 权函数 |
| --- | --- | --- |
| Gauss-Legendre | $[-1,1]$ | $1$ |
| Gauss-Chebyshev | $[-1,1]$ | $(1-x^2)^{-1/2}$ |
| Gauss-Laguerre | $[0,\infty)$ | $e^{-x}$ |
| Gauss-Hermite | $(-\infty,\infty)$ | $e^{-x^2}$ |


## Gauss-Legendre 节点和权重

Gauss-Legendre 求积近似

$$
\int_{-1}^{1} f(x)\,dx\approx \sum_{i=1}^n w_i f(x_i).
$$

节点 $x_i$ 是 Legendre 多项式 $P_n(x)$ 的零点。权重可以由 Lagrange 基函数积分得到，也可以通过 Golub-Welsch 思想由 Jacobi 矩阵的特征向量得到。

Legendre 多项式满足三项递推，因此对应的 Jacobi 矩阵是对称三对角矩阵，非对角元为

$$
\beta_k=\frac{k}{\sqrt{4k^2-1}},\qquad k=1,2,\ldots,n-1.
$$

其特征值是节点，权重为

$$
w_i=2v_{1i}^2,
$$

其中 $v_{1i}$ 是第 $i$ 个归一化特征向量的第一分量。


In [ ]:
def teaching_gauss_legendre(order):
    diagonal = np.zeros(order)
    k = np.arange(1, order, dtype=float)
    beta = k / np.sqrt(4.0 * k**2 - 1.0)
    jacobi = np.diag(diagonal)
    if order > 1:
        jacobi += np.diag(beta, 1) + np.diag(beta, -1)
    nodes, eigenvectors = np.linalg.eigh(jacobi)
    weights = 2.0 * eigenvectors[0, :] ** 2
    return nodes, weights

nodes, weights = teaching_gauss_legendre(4)
formal_nodes, formal_weights = gauss_legendre_nodes_weights(4)
print("教学版节点:", nodes)
print("正式实现节点:", formal_nodes)
print("教学版权重:", weights)
print("正式实现权重:", formal_weights)


## 代数精度验证

四点 Gauss-Legendre 公式理论上应精确积分次数不超过 7 的多项式。下面直接用单项式检查。


In [ ]:
def exact_monomial_integral_on_minus_one_one(k):
    if k % 2 == 1:
        return 0.0
    return 2.0 / (k + 1)

nodes, weights = gauss_legendre_nodes_weights(4)
for k in range(9):
    numerical = np.dot(weights, nodes**k)
    exact = exact_monomial_integral_on_minus_one_one(k)
    print(f"k={k:<2d} numerical={numerical: .16f} exact={exact: .16f} error={abs(numerical-exact):.1e}")


## 标准区间到一般区间

对一般区间 $[a,b]$，使用线性变换

$$
x=\frac{a+b}{2}+\frac{b-a}{2}t.
$$

因此

$$
\int_a^b f(x)\,dx
=\frac{b-a}{2}\int_{-1}^{1}f\left(\frac{a+b}{2}+\frac{b-a}{2}t\right)\,dt.
$$


In [ ]:
exact = math.e - 1.0 / math.e
orders = np.arange(2, 13)
gauss_errors = np.array([abs(gauss_legendre_integrate(np.exp, -1.0, 1.0, int(n)) - exact) for n in orders])
simpson_errors = np.array([abs(composite_simpson(np.exp, -1.0, 1.0, int(2 * n)) - exact) for n in orders])

print("函数计算次数近似相同的比较：")
print("n    Gauss 误差       Simpson 误差")
for n, eg, es in zip(orders, gauss_errors, simpson_errors):
    print(f"{n:<4d} {eg: .3e}    {es: .3e}")

plt.figure(figsize=(7, 4.2))
plt.semilogy(orders, gauss_errors, "o-", label="Gauss-Legendre，n 点")
plt.semilogy(orders, simpson_errors, "s-", label="复合 Simpson，约 2n+1 点")
plt.xlabel("Gauss 点数 n")
plt.ylabel("绝对误差")
plt.title("光滑函数上 Gauss-Legendre 的高效率")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 其他带权高斯公式

本轮不把所有特殊正交多项式的零点求解都写成完整通用库，但必须明确每个公式计算的积分类型。

Gauss-Chebyshev：

$$
\int_{-1}^{1}\frac{f(x)}{\sqrt{1-x^2}}\,dx
\approx \frac{\pi}{n}\sum_{k=1}^n f\left(\cos\frac{2k-1}{2n}\pi\right).
$$

Gauss-Laguerre：

$$
\int_0^\infty e^{-x}f(x)\,dx.
$$

Gauss-Hermite：

$$
\int_{-\infty}^{\infty} e^{-x^2}f(x)\,dx.
$$


In [ ]:
cheb_value = gauss_chebyshev_integrate(lambda x: x**2, order=16)
lag_value = gauss_laguerre_integrate(lambda x: x, order=16)
herm_value = gauss_hermite_integrate(lambda x: x**2, order=16)

print("Gauss-Chebyshev  int x^2/sqrt(1-x^2) dx:", cheb_value, "exact", math.pi / 2)
print("Gauss-Laguerre   int exp(-x) x dx:", lag_value, "exact", 1.0)
print("Gauss-Hermite    int exp(-x^2) x^2 dx:", herm_value, "exact", math.sqrt(math.pi) / 2)


## 适用条件、失败情形和小结

* Gauss-Legendre 对光滑函数通常比等距 Newton-Cotes 更高效。
* Gauss 型公式的节点不等距，因此不适合直接处理已经固定好的实验采样数据。
* 带权公式必须匹配积分中的权函数；把权函数漏掉会导致完全不同的问题。
* 对端点奇异、强振荡或不连续函数，高斯节点不一定自动解决困难，仍可能需要变量变换、自适应划分或专门方法。
* Gauss-Laguerre 和 Gauss-Hermite 的完整节点构造会在后续正交多项式专题中继续完善。
